# ⚗️ IRMS Analysis: Geographic Origin of Olive Oil
### FBMFOR — Food Fraud Analysis | University of Reading

This notebook loads the stable-isotope ratio data from `data/irms/irms.csv`,
pivots it from long to wide format, plots the δ²H / δ¹⁸O geographic-origin map
and a δ¹³C C3/C4 adulteration strip, and assigns each sample to the nearest
published reference region.


## 1 · Setup

In [ ]:
# ============================================================
# SETUP — INSTALL AND IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import matplotlib.patheffects as pe
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab      # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Paths ─────────────────────────────────────────────────────
if IN_COLAB:
    DATA_FILE  = Path('/content/data/irms/irms.csv')
    OUTPUT_DIR = Path('/content/data/processed')
    print("\u25b6 Running on Google Colab")
else:
    DATA_FILE  = Path('../data/irms/irms.csv')
    OUTPUT_DIR = Path('../data/processed')
    print("\u25b6 Running locally")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Download data from GitHub when running in Colab ───────────
GITHUB_RAW = "https://raw.githubusercontent.com/ggkuhnle/fbmfor-foodfraud/main"
if IN_COLAB and not DATA_FILE.exists():
    import urllib.request
    DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
    url = f"{GITHUB_RAW}/data/irms/irms.csv"
    print(f"  Downloading {url} ...")
    urllib.request.urlretrieve(url, DATA_FILE)
    print("  Done \u2713")

print(f"   Data file      : {DATA_FILE.resolve()}")
print(f"   Output dir     : {OUTPUT_DIR.resolve()}")
print("\nLibraries loaded \u2713")


## 2 · Load & reshape data

The instrument export uses a **long format**: one row per (sample, isotope) pair.

| Column | Content |
|--------|---------|
| `Sample` | Sample identifier |
| `delta` | Isotope ratio value (‰) |
| `IR` | Isotope system: `d13C`, `d2H`, or `d18O` |

We pivot to wide format so each row is one sample and each column is one isotope.
Sample-name variants (`Oil1` / `Oil 1`) are normalised automatically.


In [ ]:
# ============================================================
# LOAD AND RESHAPE IRMS DATA
# ============================================================

# ── Read raw CSV (trailing comma produces an unnamed extra column) ────────────
raw = pd.read_csv(DATA_FILE)
raw.columns = [c.strip() for c in raw.columns]

# Drop any fully-NaN columns (artifact of trailing commas in export)
raw = raw.dropna(axis=1, how='all')
raw = raw.dropna(subset=['Sample', 'delta', 'IR'])

print("Raw data:")
print(raw.to_string(index=False))
print()

# ── Normalise sample names ────────────────────────────────────────────────────
# Remove interior spaces so "Oil 1" and "Oil1" become the same key.
raw['Sample'] = raw['Sample'].str.strip().str.replace(r'\s+', '', regex=True)

# ── Normalise IR column ───────────────────────────────────────────────────────
raw['IR'] = raw['IR'].str.strip()

# ── Pivot long → wide ────────────────────────────────────────────────────────
df = (raw.pivot_table(index='Sample', columns='IR', values='delta', aggfunc='mean')
        .reset_index()
        .rename_axis(None, axis=1))

# Rename Sample → SampleID for downstream compatibility
df = df.rename(columns={'Sample': 'SampleID'})
df.columns.name = None

print("Wide-format table:")
print(df.to_string(index=False))
print()

# ── Detect available isotope columns ─────────────────────────────────────────
HAS_13C  = 'd13C'  in df.columns
HAS_2H   = 'd2H'   in df.columns
HAS_18O  = 'd18O'  in df.columns
HAS_HO   = HAS_2H and HAS_18O

iso_present = [c for c in ['d13C', 'd2H', 'd18O', 'd15N'] if c in df.columns]
print(f"Isotopes detected : {iso_present}")
print(f"H/O origin plot   : {'yes' if HAS_HO else 'no  (need d2H + d18O)'}")
print(f"C3/C4 strip       : {'yes' if HAS_13C else 'no  (need d13C)'}")


## 3 · Literature reference regions

Torres-Cobos and colleages have analysed olive oils from a range of different regions to identify Italian and non-Italian sources. The median (IQR) values are shown in the table below.

| Region   | δ${18}$O (‰)       | δ${2}$H (‰)                | δ${13}$C (‰)          |
|----------|--------------------|----------------------------|-----------------------|
| Italy    | 24.0 (23.3 -25.1)  | −144.0 (-148.8 – (-140.0)) | -29.7 (-30.1 – -29.1) |
| Apulia   | 23.7 (23.0 – 24.8) | -142.0 (-146.4 – -139.0)   | -29.9 (-30.2 – -29.5) |
| Calabria | 24.5 (23.5 – 25.9) | -141.8 (-145.0 – -135.8)   | -29.5 (-29.9 – -29.0) |
| Sicily   | 25.1 (24.4 – 25.7) | -146.5 (-150.8 – -143.7)   | -28.9 (-29.1 – -28.9) |
| Spain    | 24.0 (23.3 -25.1)  | −144.0 (-148.8 – (-140.0)) | -29.7 (-29.9 – -29.3) |
| Greece   | 24.0 (23.3 -25.1)  | −144.0 (-148.8 – (-140.0)) | -29.2 (-29.7 – -28.8) |
| Portugal | 24.0 (23.3 -25.1)  | −144.0 (-148.8 – (-140.0)) | -29.9 (-30.0 – -29.2) |
| Tunesia  | 24.0 (23.3 -25.1)  | −144.0 (-148.8 – (-140.0)) | -30.1 (-30.0 – -29.2) |
| Türkiye  | 24.0 (23.3 -25.1)  | −144.0 (-148.8 – (-140.0)) | -29.2 (-29.7 – -28.5) |


> Torres-Cobos et al. (2025) *Food Chem.* doi.org/10.1016/j.foodchem.2025.143655



In [ ]:
# ============================================================
# LITERATURE REFERENCE REGIONS
# ============================================================

REF_HO = {
    "Italy":   {"d18O_mean": 24.0, "d18O_sd": 1.8,
                "d2H_mean": -144.0, "d2H_sd": 12.0,
                "colour": "#2E86AB", "n": 242},
    "Spain":   {"d18O_mean": 26.0, "d18O_sd": 2.0,
                "d2H_mean": -140.0, "d2H_sd": 14.0,
                "colour": "#E84855", "n": 80},
    "Greece":  {"d18O_mean": 26.5, "d18O_sd": 2.2,
                "d2H_mean": -136.0, "d2H_sd": 15.0,
                "colour": "#3BB273", "n": 45},
    "Tunisia": {"d18O_mean": 27.5, "d18O_sd": 1.8,
                "d2H_mean": -128.0, "d2H_sd": 12.0,
                "colour": "#F4A261", "n": 30},
}

# δ¹³C reference for C3/C4 adulteration detection
C3_RANGE = (-32, -22)   # ‰ VPDB  — C3 oils (olive, sunflower, rapeseed)
C4_RANGE = (-16,  -9)   # ‰ VPDB  — C4 adulterants (corn oil)

print("✓ Reference data loaded")
for r, v in REF_HO.items():
    print(f"  {r:<10}  δ¹⁸O = {v['d18O_mean']:.1f} ± {v['d18O_sd']:.1f}‰  "
          f"δ²H = {v['d2H_mean']:.1f} ± {v['d2H_sd']:.1f}‰  (n={v['n']})")


## 4 · Summary statistics


In [ ]:
# ============================================================
# SUMMARY STATISTICS
# ============================================================

col_labels = {
    "d13C": "δ¹³C (‰ VPDB)",
    "d2H":  "δ²H  (‰ VSMOW)",
    "d18O": "δ¹⁸O (‰ VSMOW)",
    "d15N": "δ¹⁵N (‰ AIR)",
}

print(f"  {'Isotope':<20}  {'Mean':>8}  {'SD':>8}  {'Min':>8}  {'Max':>8}  {'N':>4}")
print("  " + "─" * 58)
for col in iso_present:
    vals = df[col].dropna()
    if len(vals) > 0:
        mean = vals.mean() if len(vals) > 1 else vals.iloc[0]
        sd   = vals.std()  if len(vals) > 1 else float('nan')
        print(f"  {col_labels.get(col, col):<20}  "
              f"{mean:8.2f}  {sd:8.2f}  {vals.min():8.2f}  {vals.max():8.2f}  {len(vals):4d}")

print()
print(df[['SampleID'] + iso_present].to_string(index=False))


## 5 · δ²H vs δ¹⁸O — Geographic Origin Plot

Both hydrogen and oxygen isotope ratios in plant-derived fats reflect
the **isotopic composition of local meteoric water**, which varies
predictably with latitude, altitude, and distance from the coast.
Italian oils (northern Mediterranean) cluster at lower δ¹⁸O than
Spanish or Tunisian oils.


In [ ]:
# ============================================================
# δ²H vs δ¹⁸O  — GEOGRAPHIC ORIGIN PLOT
# ============================================================

if not HAS_HO:
    print("Skipped — need both d2H and d18O columns.")
else:
    ELLIPSE_SD = 1.0
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_facecolor("#FAFAFA")

    # ── Reference ellipses ────────────────────────────────────────────────────
    for region, vals in REF_HO.items():
        ell = Ellipse(
            (vals["d18O_mean"], vals["d2H_mean"]),
            width=2 * ELLIPSE_SD * vals["d18O_sd"],
            height=2 * ELLIPSE_SD * vals["d2H_sd"],
            facecolor=vals["colour"], alpha=0.18,
            edgecolor=vals["colour"], linewidth=1.5, linestyle="--", zorder=1,
        )
        ax.add_patch(ell)
        ax.text(vals["d18O_mean"], vals["d2H_mean"], region,
                fontsize=9, color=vals["colour"], fontweight="bold",
                ha="center", va="center", zorder=3,
                path_effects=[pe.withStroke(linewidth=2, foreground="white")])

    # ── Sample points ─────────────────────────────────────────────────────────
    for _, row in df.iterrows():
        ax.scatter(row["d18O"], row["d2H"],
                   s=90, color="#2C3E50", edgecolors="white",
                   linewidths=0.9, zorder=5)
        ax.annotate(str(row["SampleID"]),
                    xy=(row["d18O"], row["d2H"]),
                    xytext=(row["d18O"] + 0.15, row["d2H"] + 1.5),
                    fontsize=9, color="#2C3E50", fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")],
                    zorder=6)

    # ── Decoration ───────────────────────────────────────────────────────────
    ax.set_xlabel(r"$\delta^{18}$O (‰ vs VSMOW)", fontsize=12)
    ax.set_ylabel(r"$\delta^{2}$H (‰ vs VSMOW)", fontsize=12)
    ax.set_title(r"Stable Isotope Map: $\delta^{2}$H vs $\delta^{18}$O"
                 "\nGeographic Origin of Olive Oil",
                 fontsize=13, fontweight="bold", pad=10)
    legend_handles = [
        mpatches.Patch(facecolor=v["colour"], alpha=0.5,
                       edgecolor=v["colour"], linestyle="--",
                       label=f"{r} (n={v['n']})")
        for r, v in REF_HO.items()
    ] + [plt.scatter([], [], s=70, color="#2C3E50",
                     edgecolors="white", linewidths=0.8, label="Sample")]
    ax.legend(handles=legend_handles,
              title=f"Reference regions (±{ELLIPSE_SD} SD)",
              title_fontsize=8.5, fontsize=8.5, loc="upper left", framealpha=0.9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.text(0.5, -0.10,
            "Ellipses: ±1 SD reference regions "
            "(Torres-Cobos et al. 2025; Longobardi et al. 2012).",
            transform=ax.transAxes, ha="center", fontsize=8, color="grey")
    plt.tight_layout()
    out_path = OUTPUT_DIR / 'irms_HO_origin.png'
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")


### Interpreting the δ²H / δ¹⁸O plot

- A sample plotting **inside the Italy ellipse** (lower δ¹⁸O, more negative δ²H)
  is consistent with Italian origin.
- A sample plotting among the **Spain / Greece / Tunisia** ellipses suggests
  non-Italian Mediterranean origin.
- A sample falling **outside all ellipses** warrants further investigation
  (unusual growing region, blending, or adulteration).


## 6 · δ¹³C — C3 / C4 Adulteration Strip

All genuine olive oil is a **C3 plant product** (δ¹³C ≈ −32 to −22‰).
Corn oil is a **C4 product** (δ¹³C ≈ −16 to −9‰).
Even a 5–10 % corn oil addition shifts the δ¹³C measurably towards less
negative values, making IRMS a sensitive fraud detector.


In [ ]:
# ============================================================
# δ¹³C STRIP PLOT — C3 vs C4 ADULTERATION DETECTION
# ============================================================

if not HAS_13C:
    print("Skipped — no d13C column.")
else:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.set_facecolor("#FAFAFA")

    # Reference zones
    ax.axvspan(*C3_RANGE, alpha=0.15, color="#2E86AB",
               label=f"C3 oils (olive, sunflower, rapeseed) {C3_RANGE[0]} to {C3_RANGE[1]}‰")
    ax.axvspan(*C4_RANGE, alpha=0.15, color="#E76F51",
               label=f"C4 adulterants (corn oil) {C4_RANGE[0]} to {C4_RANGE[1]}‰")

    # Samples (jittered vertically for readability)
    rng = np.random.default_rng(42)
    y_jit = rng.uniform(-0.08, 0.08, size=len(df))
    for i, (_, row) in enumerate(df.iterrows()):
        ax.scatter(row["d13C"], y_jit[i], s=80, color="#2C3E50",
                   edgecolors="white", linewidths=0.8, zorder=5)
        ax.annotate(str(row["SampleID"]),
                    xy=(row["d13C"], y_jit[i]),
                    xytext=(row["d13C"], y_jit[i] + 0.07),
                    fontsize=9, ha="center", color="#2C3E50", fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")],
                    zorder=6)

    ax.set_xlabel(r"$\delta^{13}$C (‰ vs VPDB)", fontsize=12)
    ax.set_yticks([])
    ax.set_title(r"$\delta^{13}$C Strip Plot — C3 vs C4 Oil Identification",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8.5, loc="upper left", framealpha=0.9)
    ax.spines[["top", "right", "left"]].set_visible(False)
    plt.tight_layout()
    out_path = OUTPUT_DIR / 'irms_C13_adulteration.png'
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")


## 7 · Origin Assignment (nearest centroid)

For each sample, compute the normalised Euclidean distance in the
δ²H / δ¹⁸O space to each reference centroid, scaled by the published
regional standard deviation on each axis:

$$d = \sqrt{\left(\frac{\delta^{18}O_{\text{sample}} - \delta^{18}O_{\text{ref}}}{\sigma_{18O}}\right)^2 + \left(\frac{\delta^{2}H_{\text{sample}} - \delta^{2}H_{\text{ref}}}{\sigma_{2H}}\right)^2}$$

A score **< 1** means the sample falls within ±1 SD of the regional centroid.


In [ ]:
# ============================================================
# NEAREST-CENTROID ORIGIN ASSIGNMENT
# ============================================================

if not HAS_HO:
    print("Skipped — need d2H and d18O.")
else:
    print(f"  {'SampleID':<12}  {'δ¹⁸O':>7}  {'δ²H':>8}  "
          f"{'Nearest region':<14}  {'Score':>6}  {'2nd closest':<14}  {'Score':>6}")
    print("  " + "─" * 76)

    for _, row in df.iterrows():
        x18, x2H = row["d18O"], row["d2H"]
        distances = {
            r: np.sqrt(((x18 - v["d18O_mean"]) / v["d18O_sd"])**2 +
                       ((x2H - v["d2H_mean"])  / v["d2H_sd"])**2)
            for r, v in REF_HO.items()
        }
        ranked = sorted(distances, key=distances.get)
        best, second = ranked[0], ranked[1]
        flag = ("  ⚠ ambiguous"
                if abs(distances[best] - distances[second]) < 0.3 else "")
        print(f"  {str(row['SampleID']):<12}  {x18:7.2f}  {x2H:8.2f}  "
              f"{best:<14}  {distances[best]:6.2f}  "
              f"{second:<14}  {distances[second]:6.2f}{flag}")

    print()
    print("  Score < 1.0 → within ±1 SD of regional centroid (good match)")
    print("  Score > 2.0 → outside ±2 SD (poor match — consider mislabelling)")


## 8 · Discussion Questions

1. In which reference region does each oil sample plot? Is the assignment
   unambiguous (score < 1) or borderline (score 1–2)?
2. Do the δ¹³C values confirm that both samples are **C3** plant products?
   At what δ¹³C value would you suspect corn-oil adulteration?
3. The δ²H and δ¹⁸O of plant lipids are influenced by **transpiration**
   as well as precipitation. How might a hot, dry growing season shift
   a sample away from the expected regional ellipse without indicating fraud?
4. Why is **dual-isotope** (δ²H + δ¹⁸O) assignment more reliable than
   using either isotope alone?
